# Sample Weights

This notebook will cover exercise answer.

* Exercise 4.1
* Exercise 4.2
* Exercise 4.3
* Exercise 4.4

As we go along, there will be some explanations.

More importantly, this method can be applied not just within mean-reversion strategy but also other strategies as well. Most of the functions below can be found under research/Sampling.

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import cqrlib as rs
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
dollar = pd.read_csv('../sample-data/dollar_bars.txt', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])

close = dollar['close']

In [ ]:
day_vol = rs.vol(close, span0 = 50)

# day_vol is a return (~0.55%); cs_filter diffs are price points, so scale by price level
events = rs.cs_filter(close, 
                    limit = day_vol.mean() * close.mean())

vb = rs.vert_barrier(data = close, 
                 events = events, 
                 period = 'days', 
                 freq = 1)

tb = rs.tri_barrier(data = close, 
                    events = events, 
                    trgt = day_vol, 
                    min_req = 0.002, 
                    num_threads = 3, 
                    ptSl = [1,1],
                    t1 = vb, 
                    side = None)

In [ ]:
concurrent_event = rs.num_co_events(data = close, 
                                    events = tb, 
                                    num_threads = 3)

In [ ]:
df0 = pd.DataFrame(index = concurrent_event.index).assign(volatility = day_vol, 
                                                          concurrent_event = concurrent_event )
df0[['volatility', 'concurrent_event']].plot(secondary_y='volatility', figsize=(10,8))
plt.show()

In [ ]:
fig = plt.figure(figsize=(10,8))
df1 = df0.groupby(concurrent_event, axis = 0).median() # Try piecewise estimate median
df2 = df0.groupby(concurrent_event, axis = 0).mean() # Try piecewise estimate mean
plt.plot(df0['concurrent_event'], df0['volatility'], '.',
         df1['concurrent_event'], df1['volatility'], '-',
         df2['concurrent_event'], df2['volatility'], '-.')
plt.axhline(y = 0.015, c='r', ls='--')
plt.axvline(x = 52, c='g', ls='--')
plt.show()

### Based on the multiple line plot

Where there is an increase in number of concurrent events, there are evidence of high volatility. However, volatility might not lead to high concurrent events.

### Based on Scatter plot

Before the vertical green dotted line (where the num of concurrent events is before 52), there is an obvious gradual increase which obviously meant a positive relationship.

After the vertical green dotted line, the median num of concurrent events display high fluctuations, as a result the regression line might experienced a kinked plot, if we were to plot them seperately. 

This is most likely due to rare occurance of concurrent events that experience more than 52 times.

Below the red line, most frequent and average volatility is actually below 0.015 (Maximum volatility was around 0.03).

### Conclusion

* An increase in concurrent events may be caused by volatility (weak positive relationship for most cases).
* An increase in volatility might not cause an increase in concurrent events.
* High number of concurrent events that occurs beyond a certain point is rare. Does not require high volatility.

In [ ]:
# Exercise 4.2
av_uniqueness_by_coevent = rs.wght_by_coevents(data = close, 
                        events = tb, 
                        num_threads = 3)

av_uniqueness_by_coevent

# not sure why but mine was never hitting 1 in the first column, 
# but seems to match the data freq from the book

Lagrange Multiplier tests for autocorrelation.

H0: the sequence was produced in a random manner.

H1: the sequence was not produced in a random manner.

If pval < 0.05 (Random: Accept H0).

In [ ]:
from statsmodels.sandbox.stats import diagnostic

pval = diagnostic.acorr_lm(av_uniqueness_by_coevent.squeeze(), maxlag = 1, autolag = None)[1]
print("pvalue for LM test: {0}".format(pval))

### Conclusion

**Statistical significance is the likelihood that a relationship between two or more variables is caused by something other than chance**

It is statistically significant, since it is lesser than 0.05.

In the earlier exercise, when we try to explore a potential relationship between volatility and number of concurrent events, it was concluded that they display a weak positive correlation.

The volatility that we were using was actually daily percentage change in price (daily returns) and in financial theory context financial returns are seen as "random" (considered sub-martingale).

Therefore it is only natural that average uniqueness of concurrent events would also display similar randomness characteristics.

In [ ]:
# Exercise 4.3
# We will start using ML models from sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score

In [ ]:
dollar = rs.bband_as_side(data = dollar, 
                          window = 50, 
                          width = 0.001)

dollar['volatility'] = rs.vol(dollar['close'], span0 = 50)

# volatility is a return; cs_filter diffs are price points, so scale by price level
events = rs.cs_filter(dollar['close'], 
                    limit = dollar['volatility'].mean() * dollar['close'].mean())

vb = rs.vert_barrier(data = dollar['close'], 
                 events = events, 
                 period = 'days', 
                 freq = 1)

tb = rs.tri_barrier(data = dollar['close'], 
                    events = events, 
                    trgt = dollar['volatility'], 
                    min_req = 0.002, 
                    num_threads = 3, 
                    ptSl = [0,2],
                    t1 = vb, 
                    side = dollar['side'])

mlabel = rs.meta_label(data = dollar['close'], 
                       events = tb, 
                       drop = False) # when you have a side binary, you won't have rare labels usually

In [ ]:
dollar['st_series'] = rs.fracDiff_FFD(data = dollar['close'].to_frame(),
                                      d = 0.2,
                                      thres = 1e-2)

rs.unit_root(dollar['st_series'].dropna())

In [ ]:
# use crossing averages (primary model indicators), volatility and stationary series only

dollar = dollar.reindex(mlabel.index)

#only generate stationarity series when data is continueous

dollar['st_series_cs'] = rs.fracDiff_FFD(data = dollar['close'].cumsum().to_frame(),
                                      d = 1.99999889,
                                      thres = 1e-5)

rs.unit_root(dollar['st_series_cs'].dropna())

In [ ]:
log_price = dollar['close'].apply(np.log)

dollar['st_series_log'] = rs.fracDiff_FFD(data = log_price.to_frame(),
                                          d = 0.2,
                                          thres = 1e-2)

rs.unit_root(dollar['st_series_log'].dropna())

In [ ]:
dollar['st_series_cs_log'] = rs.fracDiff_FFD(data = log_price.cumsum().to_frame(),
                                              d = 1.99999889,
                                              thres = 1e-5)

rs.unit_root(dollar['st_series_cs_log'].dropna())

In [ ]:
X = dollar.drop(['open', 'high', 'low', 'close','cum_vol', 'cum_dollar', 'cum_ticks'], axis = 1)
X.dropna(inplace = True) # we lost quite abit of data

mlabel = mlabel.reindex(X.index)
y = mlabel['bin']

n_estimators, max_depth, c_random_state = 500, 7, 42

# Random Forest Model
rf = RandomForestClassifier(max_depth = max_depth, 
                            n_estimators = n_estimators,
                            criterion = 'entropy',
                            #bootstrap=True,
                            #max_samples = av_uniqueness_by_coevent['tW'].mean()
                            oob_score = True,
                            class_weight = None, #This will be covered later
                            random_state = c_random_state)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

rf.fit(X_train, y_train.values.ravel())

print("Out-of-bag Accuracy (OOB Score): {:.6f}".format(rf.oob_score_))

rs.feat_imp(rf, X)

Our OOB score is: 0.678175

In sklearn version 0.22 they start to include max_samples, which was previously not covered.

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

>"A second and better solution is to utilise the average uniqueness..
>
> Accordingly, we could sample only a fraction out['tW'].mean() of the observations or accept a small multiple of that"
>
>   Advances in Financial Machine Learning, page 63, last paragraph.

For more information regarding Out-of-Bag Score vs Random Forest Score:

https://datascience.stackexchange.com/questions/13151/randomforestclassifier-oob-scoring-method

Optional

You may wish to run func feat_imp from rs. I included 4 different stationary series.

At this point, you should be familiar with stationarity concept and what it means to a mean-reversion strategy.

If by some chance stationarity is not considered as a key feature by random forest model, you might wish to revisit the previous steps before meta-labeling.

In the above case, it seems that using non-log stationary series has the highest feature importance rank.

In [ ]:
# Exercise 4.3b

# k-fold
no_of_folds = 5
kfold = KFold(shuffle = False, 
              random_state = 1, 
              n_splits = no_of_folds)

accuracy_array = np.zeros(no_of_folds)
i = 0
for train_index, test_index in kfold.split(X):
    # print("TRAIN:", train_index, "TEST:", test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    rf.fit(X_train, y_train.values.ravel())

    y_pred_rf = rf.predict_proba(X_test)[:, 1] #True positive only
    y_pred = rf.predict(X_test)
    accuracy_array[i] = accuracy_score(y_test, y_pred)
    i += 1
    
print("Mean KFold accuracy: {:.6f}".format(np.mean(accuracy_array)))

Mean KFold score: 0.539802

For more details onsklearn KFold:

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html

https://machinelearningmastery.com/k-fold-cross-validation/

It is obvious why OOB score is much higher than KFold score. 

KFold only splits data to sample, but it does not replace or repick them. 

Random Forest OOB actually picks and replace samples (Bootstrap = True by default) and when uniqueness of dataset is not high due to concurrent events; this will make your OOB score very much the same as In-the-Bag sample and redundant to each other.

**KFold will reflect a less bias/ more correct outcome.**

For more details:

Refer to Advances in Financial Machine Learning, page 62 - 63, section 4.5.

In [ ]:
# Exercise 4.4
# This func can be found under research/Sampling/sample_unique

wght_td = rs.wght_by_td(data = close, events = tb, num_threads = 3, td = 1.0)
wght_td0 = rs.wght_by_td(data = close, events = tb, num_threads = 3, td = 0.75)
wght_td1 = rs.wght_by_td(data = close, events = tb, num_threads = 3, td = 0.5)
wght_td2 = rs.wght_by_td(data = close, events = tb, num_threads = 3, td = 0.0)
wght_td3 = rs.wght_by_td(data = close, events = tb, num_threads = 3, td = -0.25)
wght_td4 = rs.wght_by_td(data = close, events = tb, num_threads = 3, td = -0.5)

# index is according to earlier func, which is part of the func wght_by_td
td_df = pd.DataFrame(index= av_uniqueness_by_coevent.index).assign(wght_td = wght_td,
                                                                   wght_td0 = wght_td0,
                                                                   wght_td1 = wght_td1,
                                                                   wght_td2 = wght_td2,
                                                                   wght_td3 = wght_td3,
                                                                   wght_td4 = wght_td4)


In [ ]:
td_df[['wght_td',
       'wght_td0',
       'wght_td1',
       'wght_td2',
       'wght_td3',
       'wght_td4']].plot(figsize=(10,8)) #looks correct, seems identical to the book example

#Once you go negative for time-decay factor, data samples will start to be omitted